# Pandora quickstart
Fetch NO$_2$ and HCHO for St. Edward's (AustinTX, Pandora 257), filter by quality, plot, and compute the HCHO/NO$_2$ ratio.

In [ ]:
!pip install -q git+https://github.com/<you>/pandora-tools.git   # or install from the shared drive (see README)

In [ ]:
# Mounting Drive lets the client use the group's shared-drive cache (Shareddrives/<drive>/pandora_cache).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandora_tools as pdt

client = pdt.PandoraClient()      # tz defaults to America/Chicago
client

## What's available?

In [ ]:
client.find_locations(near=(30.27, -97.74), radius_km=300)

In [ ]:
client.instruments("AustinTX")

In [ ]:
client.products("AustinTX", 257, "2026-08-01", "2026-08-31")

## Fetch (downloads once, then reads from the cache)

In [ ]:
no2, hcho = client.fetch("AustinTX", 257, "2026-08-28", products=["rnvh3", "rfuh5"])
no2.stations

In [ ]:
hcho.find_columns("formaldehyde")

In [ ]:
hcho_ok = hcho.filter_quality("hcho")        # quality flags 0, 1, 10, 11
no2_ok = no2.filter_quality("no2")
print(len(hcho), len(hcho_ok), len(no2), len(no2_ok))

## Plot

In [ ]:
def plot_series(df, col, ylabel, title, fname):
    fig, ax = plt.subplots()
    ax.plot(df["local_time"], df[col])
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=df["local_time"].dt.tz))
    ax.set_xlabel("Local time (CDT)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.savefig(fname, dpi=300, bbox_inches="tight")
    plt.show()

site = "St. Edward's (Austin, TX), August 28, 2026"
plot_series(hcho_ok, "hcho_surf_conc", "HCHO surface concentration (mol/m$^3$)", f"HCHO surface concentration, {site}", "HCHO_surface_20260828.png")
plot_series(hcho_ok, "hcho_trop_col", "HCHO tropospheric column (mol/m$^2$)", f"HCHO tropospheric column, {site}", "HCHO_trop_column_20260828.png")
plot_series(no2_ok, "no2_surf_conc", "NO$_2$ surface concentration (mol/m$^3$)", f"NO$_2$ surface concentration, {site}", "NO2_surface_20260828.png")
plot_series(no2_ok, "no2_trop_col", "NO$_2$ tropospheric column (mol/m$^2$)", f"NO$_2$ tropospheric column, {site}", "NO2_trop_column_20260828.png")

## HCHO / NO$_2$ ratio

In [ ]:
ratio = pdt.merge_products(hcho_ok, no2_ok)       # exact timestamp match; prints how many matched
ratio["hcho_no2_ratio"] = ratio["hcho_trop_col"] / ratio["no2_trop_col"]
plot_series(ratio, "hcho_no2_ratio", "HCHO to NO$_2$ ratio", f"HCHO to NO$_2$ ratio, {site}", "hcho_no2_ratio_20260828.png")

## Profiles (optional)

In [ ]:
prof = no2_ok.profiles
prof[prof["species"] == "no2"].head()